In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
import datetime
import concurrent.futures
from dateutil.relativedelta import relativedelta
import dask

In [2]:
lat_dict = {
    'full': slice(50, 25),
    'small': slice(45, 30),
    'slgt_small': slice(50, 25),
    'slgt_full': slice(50, 25)
}

lon_dict = {
    'full': slice(360-125, 360-66),
    'small': slice(360-105, 360-85),
    'slgt_small': slice(360-125, 360-66),
    'slgt_full': slice(360-125, 360-66)
}

levels_dict = {
    'full': [925, 850, 700, 500, 300],
    'small': [925, 850, 700, 500, 300],
    'slgt_small': [925, 850, 700, 500, 300],
    'slgt_full': [925, 850, 700, 500, 300]
}

time_thin_dict = {
    'full': 1,
    'small': 6,
    'slgt_small': 6,
    'slgt_full': 1,
}

space_thin_dict = {
    'full': 1,
    'small': 4,
    'slgt_small': 4,
    'slgt_full': 1
}

risk_level_dict = {
    'full': ['MDT', 'HIGH'],
    'small': ['MDT', 'HIGH'],
    'slgt_small': ['SLGT', 'ENH', 'MDT', 'HIGH'],
    'slgt_full': ['SLGT', 'ENH', 'MDT', 'HIGH']
}

pressure_var_dict = {
    'full': ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    'small': ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    'slgt_small': ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    'slgt_full': ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"]
}

surface_var_dict = {
    'full': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    'small': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    'slgt_small': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    'slgt_full': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"]
}

In [3]:
detail = 'small'

In [4]:
# --- risk days
pph = xr.load_dataset("data/raw_data/labelled_pph.nc")
missing_dates = [
    '200204250000', '200208300000', '200304150000', '200304160000',
    '200306250000', '200307270000', '200307280000', '200312280000',
    '200404140000', '200408090000', '200905280000', '201105210000',
    '202005240000', '200510240000'
]
dates_of_interest = pph["time"][pph["MAX_CAT"].isin(risk_level_dict[detail])]
dates_of_interest = dates_of_interest[dates_of_interest > "200203310000"]
dates_of_interest = dates_of_interest[~(dates_of_interest.isin(missing_dates))]
selected_days = pd.to_datetime(dates_of_interest.values, format="%Y%m%d%H%M").normalize()

In [5]:
LONG_TO_SHORT = {
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v",
    "2m_temperature": "2t",
    "2m_dewpoint_temperature": "2d",
    # "geopotential_at_surface": "z",
    # "toa_incident_solar_radiation": "tisr",
    # PRESSURE LEVEL VARS
    "geopotential": "z",
    # "potential_vorticity": "pv",
    "specific_humidity": "q",
    "temperature": "t",
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
    "vertical_velocity": "w"
}

SHORT_TO_LONG = {value: key for key, value in LONG_TO_SHORT.items()}

CHANNEL_TO_CODE = {
    "10u": 165,
    "10v": 166,
    "2t": 167,
    "2d": 168,
    "z": 129,
    "tisr": 212,

    "pv": 60,
    "q": 133,
    "t": 130,
    "u": 131,
    "v": 132,
    "w": 135
}

SURFACE_CHANNELS = ["10u", "10v", "2t", "2d", "tisr"]

In [6]:
surface_vars = [LONG_TO_SHORT[v] for v in surface_var_dict[detail]]
pressure_vars = [LONG_TO_SHORT[v] for v in pressure_var_dict[detail]]

In [7]:
LEVEL_DATA_PATH = "/glade/campaign/collections/rda/data/d633000/e5.oper.an.pl/"
SFC_DATA_PATH = "/glade/campaign/collections/rda/data/d633000/e5.oper.an.sfc/"

In [8]:
def process_code(code, year, month, month_end_day, day, time):
    # ERA5 filename logic
    if code in ['u', 'v']:
        termll025 = 'll025uv'
    else:
        termll025 = 'll025sc'

    if code in SURFACE_CHANNELS:
        path = (
            f"{SFC_DATA_PATH}{year}{month}/"
            f"e5.oper.an.sfc.128_{CHANNEL_TO_CODE[code]}_{code}."
            f"{termll025}.{year}{month}0100_{year}{month}{month_end_day}23.nc"
        )
    else:
        path = (
            f"{LEVEL_DATA_PATH}{year}{month}/"
            f"e5.oper.an.pl.128_{CHANNEL_TO_CODE[code]}_{code}."
            f"{termll025}.{year}{month}{day}00_{year}{month}{day}23.nc"
        )

    ds = xr.open_dataset(
        path,
        chunks={"time": -1}
    )

    # identify variable name
    var_name = [v for v in ds.data_vars if v != "utc_date"][0]
    da = ds[var_name]

    # ---- TIME SELECTION ----
    day_start = time.replace(hour=0, minute=0, second=0, microsecond=0)
    day_end = time.replace(hour=23, minute=0, second=0, microsecond=0)

    if code in SURFACE_CHANNELS:
        da = da.sel(time=slice(day_start, day_end))
    else:
        da = da.sel(level=levels_dict[detail])

    # ---- STANDARDIZE DIMS ----
    da = da.rename(
        latitude="lat",
        longitude="lon"
    )

    da = da.expand_dims(channel=[code])

    return da.sel(lat=lat_dict[detail], lon=lon_dict[detail]).thin({
                "lat": space_thin_dict[detail],
                "lon": space_thin_dict[detail],
                "time": time_thin_dict[detail]
            })


def get_data(time, vars):
    year = str(time.year)
    month = str(time.month).zfill(2)
    day = str(time.day).zfill(2)
    month_end_date = time + relativedelta(day=31)
    month_end_day = month_end_date.day

    dataarrays = [
        process_code(v, year, month, month_end_day, day, time)
        for v in vars
    ]

    return xr.concat(dataarrays, dim="channel")


def build_single_day_dataset(
    day,
    day0_sfc,
    day1_sfc,
    day0_pl,
    day1_pl,
):
    """
    Build ONE lazy xarray.Dataset for a single 12z–12z day.
    Assumes time is already thinned appropriately in inputs.
    """

    # ------------------------------------------------------------
    # 1. Select 12z→24z from day0 and 00z→12z from day1
    # ------------------------------------------------------------
    def select_12z_window(da0, da1):
        da0_sel = da0.sel(time=da0.time.dt.hour >= 12)
        da1_sel = da1.sel(time=da1.time.dt.hour < 12)
        return xr.concat([da0_sel, da1_sel], dim="time")

    sfc = select_12z_window(day0_sfc, day1_sfc)
    pl = select_12z_window(day0_pl,  day1_pl)

    # ------------------------------------------------------------
    # 2. Build TOD coordinate (relative to 12z)
    # ------------------------------------------------------------
    tod = (((sfc.time.dt.hour - 12) % 24).astype(int)).values

    sfc = (
        sfc
        .assign_coords(tod=("time", tod))
        .swap_dims({"time": "tod"})
        .drop_vars("time")
    )

    pl = (
        pl
        .assign_coords(tod=("time", tod))
        .swap_dims({"time": "tod"})
        .drop_vars("time")
    )

    # ------------------------------------------------------------
    # 3. Add singleton day dimension
    # ------------------------------------------------------------
    sfc = sfc.expand_dims(day=[np.datetime64(pd.Timestamp(day), "ns")])
    pl = pl.expand_dims(day=[np.datetime64(pd.Timestamp(day), "ns")])

    # ------------------------------------------------------------
    # 4. Reorder dimensions to final layout
    # ------------------------------------------------------------
    sfc = sfc.transpose("channel", "lat", "lon", "day", "tod")
    pl = pl.transpose("channel", "level", "lat", "lon", "day", "tod")

    # ------------------------------------------------------------
    # 5. Split channels into named variables (with renaming)
    # ------------------------------------------------------------
    data_vars = {}

    # Surface variables
    for ch in sfc.channel.values:
        ch = str(ch)
        if ch not in SHORT_TO_LONG:
            continue

        varname = SHORT_TO_LONG[ch]
        data_vars[varname] = (
            sfc
            .sel(channel=ch)
            .drop_vars("channel")
        )

    # Pressure-level variables
    for ch in pl.channel.values:
        ch = str(ch)
        if ch not in SHORT_TO_LONG:
            continue

        varname = SHORT_TO_LONG[ch]
        data_vars[varname] = (
            pl
            .sel(channel=ch)
            .drop_vars("channel")
        )

    # ------------------------------------------------------------
    # 6. Assemble final Dataset
    # ------------------------------------------------------------
    ds = xr.Dataset(
        data_vars=data_vars,
        coords={
            "latitude":  sfc.lat.astype("float32"),
            "longitude": sfc.lon.astype("float32"),
            "level":     pl.level.astype("int64"),
            "day":       sfc.day,
            "tod":       sfc.tod.astype("int64"),
        },
    )

    ds = ds.swap_dims({
        "lat": "latitude",
        "lon": "longitude",
    })
    ds = ds.drop_vars(["lat", "lon"])

    # ------------------------------------------------------------
    # 7. Final chunking (Zarr- & ML-friendly)
    # ------------------------------------------------------------
    ds = ds.chunk({
        "latitude":  sfc.sizes["lat"],
        "longitude": sfc.sizes["lon"],
        "level":     pl.sizes.get("level", 1),
        "day":       1,
        "tod":       sfc.sizes["tod"],
    })

    return ds

In [9]:
out_dir = Path(
    f"/glade/work/milesep/convective_outlook_ml/inputs_raw_{detail}_glade.zarr"
)
first = True
i = 0

for day in selected_days:
    print("day", i, day)
    i += 1
    next_day = day + relativedelta(day=1)
    print('0')
    day0_sfc = get_data(day, surface_vars)
    day1_sfc = get_data(next_day, surface_vars)

    day0_pl = get_data(day, pressure_vars)
    day1_pl = get_data(next_day, pressure_vars)

    print("saving just day1_pl")
    day1_pl.to_zarr('test.zarr')
    print('1')
    ds = build_single_day_dataset(
        day,
        day0_sfc,
        day1_sfc,
        day0_pl,
        day1_pl,
    )


    print('saving pl')

    pl = ds[pressure_var_dict[detail]]

    pl = pl.chunk({
        "level": -1,        # all 5 levels in one chunk
        "latitude": -1,     # full lat
        "longitude": -1,    # full lon
        "tod": -1,          # all tod for the day
        "day": 1,
    })

    with dask.config.set(scheduler="processes"):
        pl.to_zarr(
            out_dir,
            mode="w" if first else "a",
            append_dim=None if first else "day",
            consolidated=False,
        )

    print('saving sfc')
    sfc = ds[surface_var_dict[detail]]

    sfc.to_zarr(
        out_dir,
        mode="a",
        append_dim="day",
        consolidated=False,
    )



# import zarr
# zarr.consolidate_metadata(out_dir)

day 0 2002-04-07 00:00:00
0
saving just day1_pl


KeyboardInterrupt: 

In [19]:
pl

<xarray.Dataset> Size: 162kB
Dimensions:              (level: 5, latitude: 16, longitude: 21, day: 1, tod: 4)
Coordinates:
  * day                  (day) datetime64[ns] 8B 2002-04-07
  * tod                  (tod) int64 32B 0 6 12 18
  * level                (level) int64 40B 925 850 700 500 300
  * latitude             (latitude) float32 64B 45.0 44.0 43.0 ... 31.0 30.0
  * longitude            (longitude) float32 84B 255.0 256.0 ... 274.0 275.0
Data variables:
    geopotential         (level, latitude, longitude, day, tod) float32 27kB dask.array<chunksize=(5, 16, 21, 1, 4), meta=np.ndarray>
    specific_humidity    (level, latitude, longitude, day, tod) float32 27kB dask.array<chunksize=(5, 16, 21, 1, 4), meta=np.ndarray>
    temperature          (level, latitude, longitude, day, tod) float32 27kB dask.array<chunksize=(5, 16, 21, 1, 4), meta=np.ndarray>
    u_component_of_wind  (level, latitude, longitude, day, tod) float32 27kB dask.array<chunksize=(5, 16, 21, 1, 4), meta=np.ndarray>
    v_component_of_wind  (level, latitude, longitude, day, tod) float32 27kB dask.array<chunksize=(5, 16, 21, 1, 4), meta=np.ndarray>
    vertical_velocity    (level, latitude, longitude, day, tod) float32 27kB dask.array<chunksize=(5, 16, 21, 1, 4), meta=np.ndarray>